## aai-511-final-project-group1


In [0]:
# Import required libraries


## 1. Data Collection: 

Data is collected and provided to you.



In [0]:
# Import the Composer_Dataset
import os
from pathlib import Path

# Define the dataset path
dataset_path = "/Volumes/main/default/composer_dataset/Composer_Dataset/NN_midi_files_extended/"

# List all files and directories in the dataset path
try:
    files = dbutils.fs.ls(dataset_path)
    print(f"Dataset location: {dataset_path}")
    print(f"\nFound {len(files)} items in the dataset directory:\n")
    
    # Display the first 10 items to understand the structure
    for item in files[:10]:
        item_type = "DIR" if item.isDir() else "FILE"
        print(f"[{item_type}] {item.name}")
    
    if len(files) > 10:
        print(f"\n... and {len(files) - 10} more items")
        
except Exception as e:
    print(f"Error accessing dataset: {e}")


In [0]:
# Explore the contents of train, dev, and test folders
folders = ['train', 'dev', 'test']

for folder in folders:
    folder_path = f"{dataset_path}{folder}/"
    print(f"\n{'='*60}")
    print(f"Exploring {folder.upper()} folder:")
    print(f"{'='*60}")
    
    try:
        items = dbutils.fs.ls(folder_path)
        print(f"Found {len(items)} items (composers/categories):\n")
        
        # Show all items in this folder
        for item in items:
            item_type = "DIR" if item.isDir() else "FILE"
            print(f"  [{item_type}] {item.name}")
            
            # If it's a directory (composer folder), count the files inside
            if item.isDir():
                try:
                    composer_files = dbutils.fs.ls(item.path)
                    midi_files = [f for f in composer_files if f.name.endswith('.mid') or f.name.endswith('.midi')]
                    print(f"       └─ Contains {len(midi_files)} MIDI files")
                except:
                    pass
                    
    except Exception as e:
        print(f"Error accessing {folder} folder: {e}")

print(f"\n{'='*60}")
print("Dataset exploration complete!")
print(f"{'='*60}")

## 2.1 Exploratory Data Analysis (EDA)

EDA examines the statistical properties and patterns of MIDI files across 9 composers (Bach, Bartok, Byrd, Chopin, Handel, Hummel, Mendelssohn, Mozart, and Schumann), analyzing duration, tempo, complexity, and pitch characteristics to identify distinguishing features for classification. This analysis reveals data quality issues, informs feature selection for the deep learning model, and guides preprocessing decisions.



In [0]:
# Install required library for MIDI processing
%pip install pretty_midi

In [0]:
# Restart Python interpreter to apply new library installations
dbutils.library.restartPython()

In [0]:
# Install required library for MIDI processing
%pip install pretty_midi

import pretty_midi
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Initialize data structures for EDA
composer_stats = []
composers = ['bach', 'bartok', 'byrd', 'chopin', 'handel', 'hummel', 'mendelssohn', 'mozart', 'schumann']

print("Starting EDA on MIDI files...\n")
print("="*80)

# Analyze sample files from each composer (using train set)
for composer in composers:
    composer_path = f"/Volumes/main/default/composer_dataset/Composer_Dataset/NN_midi_files_extended/train/{composer}/"
    
    try:
        # Get all MIDI files for this composer
        midi_files = dbutils.fs.ls(composer_path)
        midi_files = [f for f in midi_files if f.name.endswith('.mid') or f.name.endswith('.midi')]
        
        # Analyze first 5 files from each composer for quick EDA
        files_processed = 0
        for midi_file in midi_files[:5]:
            try:
                # Read binary content using Spark (works on serverless with UC Volumes)
                import io
                binary_df = spark.read.format("binaryFile").load(midi_file.path).collect()
                if len(binary_df) == 0:
                    raise Exception("File is empty")
                
                file_content = binary_df[0].content
                
                # Load MIDI from bytes
                midi_data = pretty_midi.PrettyMIDI(io.BytesIO(file_content))
                
                # Extract features
                stats = {
                    'composer': composer,
                    'file_name': midi_file.name,
                    'duration_sec': midi_data.get_end_time(),
                    'tempo_estimate': midi_data.estimate_tempo(),
                    'num_instruments': len(midi_data.instruments)
                }
                
                # Count total notes across all instruments
                total_notes = sum([len(instrument.notes) for instrument in midi_data.instruments])
                stats['total_notes'] = total_notes
                
                # Get pitch statistics
                if midi_data.instruments:
                    pitches = [note.pitch for instrument in midi_data.instruments for note in instrument.notes]
                    if pitches:
                        stats['pitch_mean'] = np.mean(pitches)
                        stats['pitch_std'] = np.std(pitches)
                        stats['pitch_range'] = max(pitches) - min(pitches)
                    else:
                        stats['pitch_mean'] = 0
                        stats['pitch_std'] = 0
                        stats['pitch_range'] = 0
                else:
                    stats['pitch_mean'] = 0
                    stats['pitch_std'] = 0
                    stats['pitch_range'] = 0
                
                composer_stats.append(stats)
                files_processed += 1
                    
            except Exception as e:
                print(f"  ⚠ Error processing {midi_file.name}: {str(e)[:60]}...")
                continue
        
        print(f"✓ Processed {files_processed} files from {composer.upper()}")
        
    except Exception as e:
        print(f"✗ Error accessing {composer}: {e}")

print("\n" + "="*80)
print(f"Total files successfully analyzed: {len(composer_stats)}")
print("="*80)

# Create DataFrame and display summary statistics
if len(composer_stats) > 0:
    df_stats = pd.DataFrame(composer_stats)
    
    print("\nDataset Statistics Summary by Composer:")
    print("-" * 80)
    summary = df_stats.groupby('composer').agg({
        'duration_sec': ['mean', 'std'],
        'tempo_estimate': ['mean', 'std'],
        'total_notes': ['mean', 'std'],
        'pitch_mean': 'mean',
        'pitch_range': 'mean'
    }).round(2)
    
    # Reset index to show composer names as a column
    summary = summary.reset_index()
    display(summary)
    
    print("\nOverall Statistics:")
    print(f"  Average duration: {df_stats['duration_sec'].mean():.2f} seconds")
    print(f"  Average tempo: {df_stats['tempo_estimate'].mean():.2f} BPM")
    print(f"  Average notes per file: {df_stats['total_notes'].mean():.0f}")
    print(f"  Average pitch: {df_stats['pitch_mean'].mean():.2f}")
else:
    print("\n⚠ WARNING: No files were successfully processed. Check file paths and MIDI file format.")

In [0]:
# Create visualizations for EDA
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set up the plotting style
sns.set_style("whitegrid")
sns.set_palette("husl")

# Create a figure with multiple subplots
fig = plt.figure(figsize=(20, 12))

# 1. Average Duration by Composer
ax1 = plt.subplot(2, 3, 1)
df_duration = df_stats.groupby('composer')['duration_sec'].mean().sort_values(ascending=False)
sns.barplot(x=df_duration.values, y=df_duration.index, ax=ax1, palette="viridis")
ax1.set_xlabel('Average Duration (seconds)', fontsize=12)
ax1.set_ylabel('Composer', fontsize=12)
ax1.set_title('Average Duration by Composer', fontsize=14, fontweight='bold')
for i, v in enumerate(df_duration.values):
    ax1.text(v + 10, i, f'{v:.1f}s', va='center')

# 2. Average Tempo by Composer
ax2 = plt.subplot(2, 3, 2)
df_tempo = df_stats.groupby('composer')['tempo_estimate'].mean().sort_values(ascending=False)
sns.barplot(x=df_tempo.values, y=df_tempo.index, ax=ax2, palette="rocket")
ax2.set_xlabel('Average Tempo (BPM)', fontsize=12)
ax2.set_ylabel('Composer', fontsize=12)
ax2.set_title('Average Tempo by Composer', fontsize=14, fontweight='bold')
for i, v in enumerate(df_tempo.values):
    ax2.text(v + 3, i, f'{v:.1f}', va='center')

# 3. Average Total Notes by Composer
ax3 = plt.subplot(2, 3, 3)
df_notes = df_stats.groupby('composer')['total_notes'].mean().sort_values(ascending=False)
sns.barplot(x=df_notes.values, y=df_notes.index, ax=ax3, palette="mako")
ax3.set_xlabel('Average Total Notes', fontsize=12)
ax3.set_ylabel('Composer', fontsize=12)
ax3.set_title('Average Total Notes by Composer', fontsize=14, fontweight='bold')
for i, v in enumerate(df_notes.values):
    ax3.text(v + 200, i, f'{int(v)}', va='center')

# 4. Box Plot - Duration Distribution
ax4 = plt.subplot(2, 3, 4)
sns.boxplot(data=df_stats, y='composer', x='duration_sec', ax=ax4, palette="Set2")
ax4.set_xlabel('Duration (seconds)', fontsize=12)
ax4.set_ylabel('Composer', fontsize=12)
ax4.set_title('Duration Distribution by Composer', fontsize=14, fontweight='bold')

# 5. Box Plot - Tempo Distribution
ax5 = plt.subplot(2, 3, 5)
sns.boxplot(data=df_stats, y='composer', x='tempo_estimate', ax=ax5, palette="Set3")
ax5.set_xlabel('Tempo (BPM)', fontsize=12)
ax5.set_ylabel('Composer', fontsize=12)
ax5.set_title('Tempo Distribution by Composer', fontsize=14, fontweight='bold')

# 6. Scatter Plot - Pitch Mean vs Range
ax6 = plt.subplot(2, 3, 6)
for composer in df_stats['composer'].unique():
    composer_data = df_stats[df_stats['composer'] == composer]
    ax6.scatter(composer_data['pitch_mean'], composer_data['pitch_range'], 
                label=composer, s=100, alpha=0.7)
ax6.set_xlabel('Pitch Mean', fontsize=12)
ax6.set_ylabel('Pitch Range', fontsize=12)
ax6.set_title('Pitch Mean vs Range by Composer', fontsize=14, fontweight='bold')
ax6.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Second figure - Correlation heatmap and additional analysis
fig2 = plt.figure(figsize=(16, 6))

# 1. Correlation Heatmap
ax7 = plt.subplot(1, 2, 1)
numeric_cols = ['duration_sec', 'tempo_estimate', 'total_notes', 'pitch_mean', 'pitch_std', 'pitch_range']
corr_matrix = df_stats[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax7)
ax7.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')

# 2. Violin Plot - Total Notes Distribution
ax8 = plt.subplot(1, 2, 2)
sns.violinplot(data=df_stats, y='composer', x='total_notes', ax=ax8, palette="muted")
ax8.set_xlabel('Total Notes', fontsize=12)
ax8.set_ylabel('Composer', fontsize=12)
ax8.set_title('Total Notes Distribution by Composer', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("EDA Visualizations Complete!")
print("="*80)
print("\nKey Observations:")
print(f"  • Longest pieces: {df_duration.index[0].title()} ({df_duration.values[0]:.1f}s avg)")
print(f"  • Fastest tempo: {df_tempo.index[0].title()} ({df_tempo.values[0]:.1f} BPM avg)")
print(f"  • Most complex: {df_notes.index[0].title()} ({int(df_notes.values[0])} notes avg)")
print(f"  • Most correlated features: Duration & Total Notes (r={corr_matrix.loc['duration_sec', 'total_notes']:.2f})")

## 2.2 Data Pre-processing: 

Convert the musical scores into a format suitable for deep learning models. This involves converting the musical scores into MIDI files and applying data augmentation techniques.

In [0]:
# Data Pre-Processing: Convert MIDI to sequences and apply augmentation
import pretty_midi
import numpy as np
import pandas as pd
from collections import Counter
import io
import random

print("="*80)
print("MIDI DATA PRE-PROCESSING")
print("="*80)

# Configuration
SEQUENCE_LENGTH = 100  # Number of notes in each sequence
SAMPLING_RATE = 4      # Notes per beat (for time quantization)

# Create composer to label mapping
composers = ['bach', 'bartok', 'byrd', 'chopin', 'handel', 'hummel', 'mendelssohn', 'mozart', 'schumann']
composer_to_label = {composer: idx for idx, composer in enumerate(composers)}
label_to_composer = {idx: composer for composer, idx in composer_to_label.items()}

print(f"\nComposer Label Mapping:")
for composer, label in composer_to_label.items():
    print(f"  {label}: {composer.title()}")

def extract_notes_from_midi(midi_path):
    """
    Extract note sequences from MIDI file.
    Returns list of (pitch, start_time, duration) tuples.
    """
    try:
        # Read MIDI file using Spark binary reader
        binary_df = spark.read.format("binaryFile").load(midi_path).collect()
        if len(binary_df) == 0:
            return None
        
        file_content = binary_df[0].content
        midi_data = pretty_midi.PrettyMIDI(io.BytesIO(file_content))
        
        notes = []
        for instrument in midi_data.instruments:
            if not instrument.is_drum:  # Skip drum tracks
                for note in instrument.notes:
                    notes.append({
                        'pitch': note.pitch,
                        'start': note.start,
                        'end': note.end,
                        'duration': note.end - note.start,
                        'velocity': note.velocity
                    })
        
        # Sort notes by start time
        notes = sorted(notes, key=lambda x: x['start'])
        return notes
        
    except Exception as e:
        return None

def notes_to_sequences(notes, sequence_length=100):
    """
    Convert notes to fixed-length sequences.
    Each note is represented as [pitch, duration, velocity].
    """
    if not notes or len(notes) < sequence_length:
        return []
    
    sequences = []
    for i in range(0, len(notes) - sequence_length + 1, sequence_length // 2):  # 50% overlap
        seq = notes[i:i + sequence_length]
        # Normalize features
        sequence_array = np.array([
            [n['pitch'] / 127.0,           # Normalize pitch to [0, 1]
             min(n['duration'], 4.0) / 4.0, # Normalize duration (cap at 4 seconds)
             n['velocity'] / 127.0]         # Normalize velocity to [0, 1]
            for n in seq
        ])
        sequences.append(sequence_array)
    
    return sequences

def augment_pitch_shift(notes, semitones):
    """
    Augment by shifting pitch up or down by semitones.
    """
    augmented = []
    for note in notes:
        new_pitch = note['pitch'] + semitones
        # Keep within MIDI range [0, 127]
        if 0 <= new_pitch <= 127:
            aug_note = note.copy()
            aug_note['pitch'] = new_pitch
            augmented.append(aug_note)
        else:
            augmented.append(note)
    return augmented

def augment_time_stretch(notes, factor):
    """
    Augment by stretching/compressing time.
    """
    augmented = []
    for note in notes:
        aug_note = note.copy()
        aug_note['start'] = note['start'] * factor
        aug_note['end'] = note['end'] * factor
        aug_note['duration'] = note['duration'] * factor
        augmented.append(aug_note)
    return augmented

def process_dataset_split(split_name, apply_augmentation=False):
    """
    Process a dataset split (train/dev/test) and extract sequences.
    """
    print(f"\n{'='*80}")
    print(f"Processing {split_name.upper()} Split")
    print(f"{'='*80}")
    
    all_sequences = []
    all_labels = []
    
    for composer in composers:
        composer_path = f"/Volumes/main/default/composer_dataset/Composer_Dataset/NN_midi_files_extended/{split_name}/{composer}/"
        
        try:
            midi_files = dbutils.fs.ls(composer_path)
            midi_files = [f for f in midi_files if f.name.endswith('.mid') or f.name.endswith('.midi')]
            
            composer_sequences = 0
            for midi_file in midi_files:
                notes = extract_notes_from_midi(midi_file.path)
                
                if notes:
                    # Extract sequences from original notes
                    sequences = notes_to_sequences(notes, SEQUENCE_LENGTH)
                    
                    if sequences:
                        all_sequences.extend(sequences)
                        all_labels.extend([composer_to_label[composer]] * len(sequences))
                        composer_sequences += len(sequences)
                    
                    # Apply augmentation only for training set
                    if apply_augmentation and split_name == 'train':
                        # Pitch shift augmentation (+/- 2 semitones)
                        for shift in [-2, 2]:
                            aug_notes = augment_pitch_shift(notes, shift)
                            aug_sequences = notes_to_sequences(aug_notes, SEQUENCE_LENGTH)
                            if aug_sequences:
                                all_sequences.extend(aug_sequences)
                                all_labels.extend([composer_to_label[composer]] * len(aug_sequences))
                        
                        # Time stretch augmentation (0.9x and 1.1x speed)
                        for factor in [0.9, 1.1]:
                            aug_notes = augment_time_stretch(notes, factor)
                            aug_sequences = notes_to_sequences(aug_notes, SEQUENCE_LENGTH)
                            if aug_sequences:
                                all_sequences.extend(aug_sequences)
                                all_labels.extend([composer_to_label[composer]] * len(aug_sequences))
            
            print(f"  {composer.title()}: {composer_sequences} sequences from {len(midi_files)} files")
            
        except Exception as e:
            print(f"  Error processing {composer}: {e}")
    
    return np.array(all_sequences), np.array(all_labels)

# Process all splits
print("\n" + "="*80)
print("EXTRACTING SEQUENCES FROM ALL SPLITS")
print("="*80)

X_train, y_train = process_dataset_split('train', apply_augmentation=True)
X_dev, y_dev = process_dataset_split('dev', apply_augmentation=False)
X_test, y_test = process_dataset_split('test', apply_augmentation=False)

# Print summary statistics
print("\n" + "="*80)
print("DATA PRE-PROCESSING COMPLETE")
print("="*80)
print(f"\nDataset Summary:")
print(f"  Training:")
print(f"    Shape: {X_train.shape}")
print(f"    Sequences: {len(X_train)}")
print(f"    Augmented: Yes (pitch shift + time stretch)")
print(f"\n  Development:")
print(f"    Shape: {X_dev.shape}")
print(f"    Sequences: {len(X_dev)}")
print(f"    Augmented: No")
print(f"\n  Test:")
print(f"    Shape: {X_test.shape}")
print(f"    Sequences: {len(X_test)}")
print(f"    Augmented: No")

print(f"\nSequence Configuration:")
print(f"  Sequence Length: {SEQUENCE_LENGTH} notes")
print(f"  Features per note: 3 (pitch, duration, velocity)")
print(f"  Input shape per sequence: ({SEQUENCE_LENGTH}, 3)")

print(f"\nClass Distribution:")
print(f"  Training set:")
train_dist = Counter(y_train)
for label in sorted(train_dist.keys()):
    print(f"    {label_to_composer[label].title()}: {train_dist[label]} sequences")

print("\n" + "="*80)

3. Feature Extraction: Extract features from the MIDI files, such as notes, chords, and tempo, using music analysis tools.

In [0]:
# Feature Extraction: Extract advanced musical features from MIDI files
import pretty_midi
import numpy as np
import pandas as pd
from collections import Counter
import io
import warnings

# Define composers list (needed for feature extraction)
composers = ['bach', 'bartok', 'byrd', 'chopin', 'handel', 'hummel', 'mendelssohn', 'mozart', 'schumann']

print("="*80)
print("ADVANCED MUSICAL FEATURE EXTRACTION")
print("="*80)

# ============================================================================
# CONFIGURATION
# ============================================================================

# Time quantization settings
FRAME_RATE = 16  # Hz - frames per second (higher = finer temporal resolution)
STEPS_PER_BEAT = 4  # Alternative: steps per beat quantization

# Pitch range settings (standard piano range)
MIN_MIDI_PITCH = 21   # A0 - lowest note on standard piano
MAX_MIDI_PITCH = 108  # C8 - highest note on standard piano
PITCH_RANGE = MAX_MIDI_PITCH - MIN_MIDI_PITCH + 1  # 88 keys

# Feature extraction modes
EXTRACT_PIANO_ROLL = True      # For CNN models
EXTRACT_NOTE_SEQUENCE = True   # For LSTM/Transformer models
EXTRACT_CHORDS = True          # Chord detection
EXTRACT_TEMPO_FEATURES = True  # Tempo and rhythm features

print(f"\nConfiguration:")
print(f"  Frame Rate: {FRAME_RATE} Hz")
print(f"  Pitch Range: {MIN_MIDI_PITCH}-{MAX_MIDI_PITCH} ({PITCH_RANGE} keys)")
print(f"  Extraction Modes: Piano Roll={EXTRACT_PIANO_ROLL}, Sequences={EXTRACT_NOTE_SEQUENCE}")

# ============================================================================
# FEATURE EXTRACTION FUNCTIONS
# ============================================================================

def extract_piano_roll(midi_data, frame_rate=16):
    """
    Extract piano roll representation (2D matrix) from MIDI data.
    
    Output shape: [time_steps, pitch_range]
    - Suitable for CNN input (can be treated as 1-channel image)
    - Each row represents a time frame
    - Each column represents a pitch (key on piano)
    - Cell values: velocity (0-127) or binary (0/1)
    
    Args:
        midi_data: PrettyMIDI object
        frame_rate: Frames per second for time quantization
    
    Returns:
        piano_roll: numpy array of shape [time_steps, pitch_range]
        metadata: dict with duration, tempo, etc.
    """
    try:
        # Get piano roll using pretty_midi's built-in function
        # Returns shape: [128, time_steps] - need to transpose and crop
        piano_roll_full = midi_data.get_piano_roll(fs=frame_rate)
        
        # Crop to our pitch range and transpose to [time_steps, pitches]
        piano_roll = piano_roll_full[MIN_MIDI_PITCH:MAX_MIDI_PITCH+1, :].T
        
        # Normalize velocities to [0, 1]
        piano_roll = piano_roll / 127.0
        
        # Extract metadata
        metadata = {
            'duration': midi_data.get_end_time(),
            'tempo': midi_data.estimate_tempo(),
            'time_steps': piano_roll.shape[0],
            'frame_rate': frame_rate
        }
        
        return piano_roll, metadata
        
    except Exception as e:
        warnings.warn(f"Error extracting piano roll: {str(e)}")
        return None, None

def extract_note_events(midi_data):
    """
    Extract note events as a sequence (for LSTM/Transformer models).
    
    Output: List of note events with timing information
    - Each event: [pitch, velocity, start_time, duration]
    - Chronologically ordered
    - Suitable for sequence-to-sequence models
    
    Args:
        midi_data: PrettyMIDI object
    
    Returns:
        note_events: list of dicts with note information
        note_density: notes per second
    """
    try:
        note_events = []
        
        for instrument in midi_data.instruments:
            if not instrument.is_drum:
                for note in instrument.notes:
                    note_events.append({
                        'pitch': note.pitch,
                        'velocity': note.velocity,
                        'start': note.start,
                        'end': note.end,
                        'duration': note.end - note.start,
                        'instrument': instrument.program
                    })
        
        # Sort by start time
        note_events = sorted(note_events, key=lambda x: x['start'])
        
        # Calculate note density
        duration = midi_data.get_end_time()
        note_density = len(note_events) / duration if duration > 0 else 0
        
        return note_events, note_density
        
    except Exception as e:
        warnings.warn(f"Error extracting note events: {str(e)}")
        return None, 0

def detect_chords(midi_data, window_size=0.05):
    """
    Detect chords (simultaneous notes) in MIDI data.
    
    Args:
        midi_data: PrettyMIDI object
        window_size: Time window (seconds) to consider notes as simultaneous
    
    Returns:
        chords: list of chord events (list of pitches at each time)
        polyphony_stats: statistics about polyphony
    """
    try:
        # Get all notes across all instruments
        all_notes = []
        for instrument in midi_data.instruments:
            if not instrument.is_drum:
                all_notes.extend(instrument.notes)
        
        # Sort by start time
        all_notes = sorted(all_notes, key=lambda x: x.start)
        
        chords = []
        chord_sizes = []
        
        i = 0
        while i < len(all_notes):
            # Find all notes that start within the window
            current_time = all_notes[i].start
            chord_notes = []
            
            j = i
            while j < len(all_notes) and all_notes[j].start <= current_time + window_size:
                chord_notes.append(all_notes[j].pitch)
                j += 1
            
            if len(chord_notes) > 1:  # Only record if it's actually a chord
                chords.append({
                    'time': current_time,
                    'pitches': sorted(set(chord_notes)),
                    'size': len(set(chord_notes))
                })
                chord_sizes.append(len(set(chord_notes)))
            
            i = j if j > i else i + 1
        
        polyphony_stats = {
            'total_chords': len(chords),
            'avg_chord_size': np.mean(chord_sizes) if chord_sizes else 0,
            'max_polyphony': max(chord_sizes) if chord_sizes else 0
        }
        
        return chords, polyphony_stats
        
    except Exception as e:
        warnings.warn(f"Error detecting chords: {str(e)}")
        return None, None

def extract_tempo_features(midi_data):
    """
    Extract tempo and rhythmic features.
    
    Args:
        midi_data: PrettyMIDI object
    
    Returns:
        tempo_features: dict with tempo-related features
    """
    try:
        tempo_features = {
            'estimated_tempo': midi_data.estimate_tempo(),
            'tempo_changes': len(midi_data.get_tempo_changes()[0]),
            'time_signature_changes': len(midi_data.time_signature_changes)
        }
        
        # Get tempo change times and values if they exist
        tempo_times, tempo_values = midi_data.get_tempo_changes()
        if len(tempo_values) > 0:
            tempo_features['min_tempo'] = float(np.min(tempo_values))
            tempo_features['max_tempo'] = float(np.max(tempo_values))
            tempo_features['tempo_variance'] = float(np.var(tempo_values))
        else:
            tempo_features['min_tempo'] = tempo_features['estimated_tempo']
            tempo_features['max_tempo'] = tempo_features['estimated_tempo']
            tempo_features['tempo_variance'] = 0.0
        
        return tempo_features
        
    except Exception as e:
        warnings.warn(f"Error extracting tempo features: {str(e)}")
        return None

def tokenize_for_transformer(note_events, vocab_size=128):
    """
    Convert note events to token sequence for Transformer models.
    
    Args:
        note_events: list of note event dictionaries
        vocab_size: size of token vocabulary (default: 128 for MIDI pitches)
    
    Returns:
        tokens: list of token indices
        token_times: corresponding timestamps for positional encoding
    """
    try:
        tokens = []
        token_times = []
        
        for event in note_events:
            # Use pitch as token (can be extended with velocity, duration tokens)
            token = event['pitch']
            if token < vocab_size:
                tokens.append(token)
                token_times.append(event['start'])
        
        return tokens, token_times
        
    except Exception as e:
        warnings.warn(f"Error tokenizing for transformer: {str(e)}")
        return None, None

def extract_all_features(midi_path):
    """
    Master function to extract all features from a MIDI file.
    
    Args:
        midi_path: Path to MIDI file (Unity Catalog Volume path)
    
    Returns:
        features: dict containing all extracted features
        success: boolean indicating successful extraction
    """
    try:
        # Read MIDI file using Spark binary reader (for UC Volumes)
        binary_df = spark.read.format("binaryFile").load(midi_path).collect()
        if len(binary_df) == 0:
            return None, False
        
        file_content = binary_df[0].content
        midi_data = pretty_midi.PrettyMIDI(io.BytesIO(file_content))
        
        features = {}
        
        # Extract piano roll (for CNN)
        if EXTRACT_PIANO_ROLL:
            piano_roll, metadata = extract_piano_roll(midi_data, FRAME_RATE)
            if piano_roll is not None:
                features['piano_roll'] = piano_roll
                features['piano_roll_metadata'] = metadata
        
        # Extract note sequences (for LSTM/Transformer)
        if EXTRACT_NOTE_SEQUENCE:
            note_events, note_density = extract_note_events(midi_data)
            if note_events is not None:
                features['note_events'] = note_events
                features['note_density'] = note_density
                
                # Tokenize for transformer
                tokens, token_times = tokenize_for_transformer(note_events)
                if tokens is not None:
                    features['transformer_tokens'] = tokens
                    features['transformer_times'] = token_times
        
        # Extract chord information
        if EXTRACT_CHORDS:
            chords, polyphony_stats = detect_chords(midi_data)
            if chords is not None:
                features['chords'] = chords
                features['polyphony_stats'] = polyphony_stats
        
        # Extract tempo features
        if EXTRACT_TEMPO_FEATURES:
            tempo_features = extract_tempo_features(midi_data)
            if tempo_features is not None:
                features['tempo_features'] = tempo_features
        
        # Add basic metadata
        features['duration'] = midi_data.get_end_time()
        features['n_instruments'] = len(midi_data.instruments)
        
        return features, True
        
    except Exception as e:
        warnings.warn(f"Error processing {midi_path}: {str(e)[:100]}")
        return None, False

# ============================================================================
# BATCH FEATURE EXTRACTION
# ============================================================================

def extract_features_from_dataset(composers, split='train', max_files_per_composer=5):
    """
    Extract features from multiple MIDI files in the dataset.
    
    Args:
        composers: list of composer names
        split: 'train', 'dev', or 'test'
        max_files_per_composer: limit files for demonstration (None = all)
    
    Returns:
        extracted_features: dict mapping file paths to feature dicts
        stats: summary statistics
    """
    print(f"\n{'='*80}")
    print(f"Extracting Features from {split.upper()} Split")
    print(f"{'='*80}")
    
    extracted_features = {}
    stats = {
        'total_files': 0,
        'successful': 0,
        'failed': 0,
        'by_composer': {}
    }
    
    for composer in composers:
        composer_path = f"/Volumes/main/default/composer_dataset/Composer_Dataset/NN_midi_files_extended/{split}/{composer}/"
        
        try:
            midi_files = dbutils.fs.ls(composer_path)
            midi_files = [f for f in midi_files if f.name.endswith('.mid') or f.name.endswith('.midi')]
            
            # Limit files for demonstration
            if max_files_per_composer is not None:
                midi_files = midi_files[:max_files_per_composer]
            
            composer_success = 0
            composer_failed = 0
            
            for midi_file in midi_files:
                stats['total_files'] += 1
                
                features, success = extract_all_features(midi_file.path)
                
                if success:
                    extracted_features[midi_file.path] = features
                    stats['successful'] += 1
                    composer_success += 1
                else:
                    stats['failed'] += 1
                    composer_failed += 1
            
            stats['by_composer'][composer] = {
                'successful': composer_success,
                'failed': composer_failed
            }
            
            print(f"  {composer.title()}: {composer_success}/{composer_success + composer_failed} files processed")
            
        except Exception as e:
            print(f"  Error accessing {composer}: {e}")
    
    return extracted_features, stats

# ============================================================================
# EXECUTE FULL FEATURE EXTRACTION (ALL FILES)
# ============================================================================

print("\n" + "="*80)
print("FULL FEATURE EXTRACTION - ALL FILES")
print("="*80)
print("\nThis will extract features from ALL files in train, dev, and test splits.")
print("Features will be stored in memory for immediate use.")
print("Processing may take several minutes...\n")

# Store all extracted features
all_features = {}
all_stats = {}

# Process each split
for split in ['train', 'dev', 'test']:
    print("\n" + "="*80)
    print(f"Processing {split.upper()} Split")
    print("="*80)
    
    # Extract features from ALL files (max_files_per_composer=None)
    features_dict, extraction_stats = extract_features_from_dataset(
        composers=composers,
        split=split,
        max_files_per_composer=None  # Process ALL files
    )
    
    # Store results
    all_features[split] = features_dict
    all_stats[split] = extraction_stats
    
    # Display statistics
    print(f"\n{split.upper()} Split Statistics:")
    print(f"  Total files processed: {extraction_stats['total_files']}")
    print(f"  Successful: {extraction_stats['successful']}")
    print(f"  Failed: {extraction_stats['failed']}")
    print(f"  Success Rate: {extraction_stats['successful']/extraction_stats['total_files']*100:.1f}%")

# ============================================================================
# SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("FEATURE EXTRACTION COMPLETE - ALL SPLITS")
print("="*80)

print("\nOverall Statistics:")
print("-" * 80)
total_processed = sum(stats['total_files'] for stats in all_stats.values())
total_successful = sum(stats['successful'] for stats in all_stats.values())
total_failed = sum(stats['failed'] for stats in all_stats.values())

print(f"  Total files processed across all splits: {total_processed}")
print(f"  Total successful: {total_successful}")
print(f"  Total failed: {total_failed}")
print(f"  Overall success rate: {total_successful/total_processed*100:.1f}%")

print("\nBreakdown by Split:")
print("-" * 80)
for split in ['train', 'dev', 'test']:
    stats = all_stats[split]
    print(f"  {split.upper():5s}: {stats['successful']:4d} files successfully processed")

print("\nBreakdown by Composer (TRAIN split):")
print("-" * 80)
for composer, stats in all_stats['train']['by_composer'].items():
    print(f"  {composer.title():12s}: {stats['successful']:3d} successful, {stats['failed']:2d} failed")

# Show sample features from first file in training set
if len(all_features['train']) > 0:
    sample_path = list(all_features['train'].keys())[0]
    sample_features = all_features['train'][sample_path]
    
    print(f"\n" + "="*80)
    print(f"SAMPLE FEATURE SET (First Training File)")
    print(f"="*80)
    print(f"File: {sample_path.split('/')[-1]}")
    print(f"\nExtracted Features:")
    
    if 'piano_roll' in sample_features:
        pr = sample_features['piano_roll']
        print(f"  Piano Roll Shape: {pr.shape} (time_steps × pitches)")
        print(f"  Duration: {sample_features['piano_roll_metadata']['duration']:.2f}s")
        print(f"  Active Notes: {np.count_nonzero(pr)} / {pr.size} ({np.count_nonzero(pr)/pr.size*100:.1f}% density)")
    
    if 'note_events' in sample_features:
        print(f"  Note Events: {len(sample_features['note_events'])} notes")
        print(f"  Note Density: {sample_features['note_density']:.2f} notes/second")
    
    if 'transformer_tokens' in sample_features:
        print(f"  Transformer Tokens: {len(sample_features['transformer_tokens'])} tokens")
    
    if 'polyphony_stats' in sample_features:
        ps = sample_features['polyphony_stats']
        print(f"  Chord Detection: {ps['total_chords']} chords")
        print(f"  Avg Chord Size: {ps['avg_chord_size']:.2f} notes")
        print(f"  Max Polyphony: {ps['max_polyphony']} simultaneous notes")
    
    if 'tempo_features' in sample_features:
        tf = sample_features['tempo_features']
        print(f"  Estimated Tempo: {tf['estimated_tempo']:.1f} BPM")
        print(f"  Tempo Range: {tf['min_tempo']:.1f} - {tf['max_tempo']:.1f} BPM")

print("\n" + "="*80)
print("READY FOR MODEL TRAINING")
print("="*80)
print("\nFeature Formats Available:")
print("  1. Piano Roll (2D) - For CNN models")
print("     Shape: [time_steps, 88] - Ready for Conv2D layers")
print("  2. Note Sequences (1D) - For LSTM models")
print("     Event-based sequence with pitch, velocity, timing")
print("  3. Token Sequences (1D) - For Transformer models")
print("     Integer tokens with positional timestamps")
print("  4. Additional: Chords, Tempo, Polyphony features")

print("\nFeatures stored in memory:")
print("  all_features['train'] - Training set features (369 files)")
print("  all_features['dev']   - Validation set features (35 files)")
print("  all_features['test']  - Test set features (35 files)")
print("\n" + "="*80)

4. Model Building: Develop a deep learning model using LSTM and CNN architectures to classify the musical scores according to the composer.

In [0]:
# 

5. Model Training: Train the deep learning model using the pre-processed and feature-extracted data.

In [0]:
# 

6. Model Evaluation: Evaluate the performance of the deep learning model using accuracy, precision, and recall metrics.

In [0]:
# 

7. Model Optimization: Optimize the deep learning model by fine-tuning hyperparameters.

In [0]:
# 